The notebook demonstrates:
1. how to render our data with our blender rendering script.
2. basic usage of plibs (eg, to visualize rgbd, point cloud, etc)
3. how to save the rendered data into a tar that can be used by our dataloader.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import numpy as np
import open3d as o3d

from lito.preprocess_scripts import tar_utils
from plibs import data_utils, structures, utils

# assume you run the notebook in the notebooks folder
repo_root = os.path.abspath("..")

In [ ]:
# download bunny mesh
mesh_dir = os.path.join(repo_root, "assets", "mesh")
mesh_filename = os.path.join(mesh_dir, "bunny.obj")

if not os.path.exists(mesh_filename):
    mesh_url = "http://kunzhou.net/tex-models/bunny.zip"
    os.makedirs(mesh_dir, exist_ok=True)
    cmd = f"cd {mesh_dir}; wget {mesh_url}; unzip bunny.zip"
    os.system(cmd)

assert os.path.exists(mesh_filename), f"{mesh_filename} does not exist"

# mesh_filename = os.path.join(repo_root, "libraries/blender_rendering/assets/bunny.glb")

# toy settings
num_regular_images = 32  # in paper we render 150
num_random_images = 5  # in paper we render 100
num_cond_images = 2  # num_cond_images = 32
num_gen_eval_images = 3  # in paper we render 150


assert os.path.exists(mesh_filename), f"{mesh_filename} does not exist"

# render with blender (which box-normalizes the mesh to [-1, 1])
out_root_dir = "render_data_results"
out_blender_dir = os.path.join(out_root_dir, "blender_render_results")
out_dict = data_utils.render_multi_mesh_sample(
    out_dir=out_blender_dir,
    mesh_filenames=[mesh_filename],
    min_mesh_scale=1,
    max_mesh_scale=1,
    # light
    light_mode="diffuse",  # "random"
    # circular camera (no anti-aliasing)
    num_regular_images=num_regular_images,
    fov=40.0,  # degree
    circular_radius=3.5,  # meter
    regular_width_px=518,
    regular_height_px=518,
    # random camera (with anti-aliasing)
    num_random_images=num_random_images,
    min_fov=20.0,  # degree
    max_fov=40.0,  # degree
    min_random_radius=1.5,
    max_random_radius=3.5,
    random_lookat_r=0.25,
    random_width_px=518,
    random_height_px=518,
    # cond camera
    num_gen_eval_images=num_gen_eval_images,
    num_cond_images=num_cond_images,
    cond_width_px=518,
    cond_height_px=518,
    gen_eval_width_px=518,
    gen_eval_height_px=518,
    # misc
    read_result_to_rgbd=True,
    overwrite=True,
    max_memory_gb=None,
    timeout=None,
    blender_device="CPU",  # "GPU"
    seed=42,
    blender_exe=None,
    render_gen=True,
    rotate_objs=False,
    adjust_exposure=True,
    exposure_cam_type="regular",
)
config_dict = out_dict["config_dict"]
rgbd_regular: structures.RGBDImage = out_dict["rgbd_regular"]  # (num_frames, num_regular_views, h_regular, w_regular)
rgbd_random: structures.RGBDImage = out_dict["rgbd_random"]  # (num_frames, num_random_views, h_random, w_random)
rgbd_cond: structures.RGBDImage = out_dict["rgbd_cond"]  # (num_frames, num_cond_views, h_random, w_random)
rgbd_gen_eval: structures.RGBDImage = out_dict["rgbd_gen_eval"]  # (num_frames, num_gen_eval_views, h_random, w_random)

In [ ]:
# (optional) backproject the rgbd images for visualization
pcd_regular: structures.PointCloud = rgbd_regular.get_pcd(subsample=4)
pcd_random: structures.PointCloud = rgbd_random.get_pcd(subsample=4)
pcd_cond: structures.PointCloud = rgbd_cond.get_pcd(subsample=4)
pcd_gen_eval: structures.PointCloud = rgbd_gen_eval.get_pcd(subsample=4)

# get open3d point cloud
o3d_pcd_random: o3d.geometry.PointCloud = pcd_random.get_o3d_pcds()[0]
o3d_pcd_regular: o3d.geometry.PointCloud = pcd_regular.get_o3d_pcds()[0]
o3d_pcd_cond: o3d.geometry.PointCloud = pcd_cond.get_o3d_pcds()[0]
o3d_pcd_gen_eval: o3d.geometry.PointCloud = pcd_gen_eval.get_o3d_pcds()[0]

In [ ]:
# (optional) visualize the backprojected points (they should align)
world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1)
utils.visualize_mesh_sequence(
    [
        o3d_pcd_regular,
        o3d_pcd_random,
        o3d_pcd_cond,
        o3d_pcd_gen_eval,
    ],
    static_meshes=[world_frame],
)

In [ ]:
# save to a local dir
data_dir = os.path.join(out_root_dir, "rgbd_results")
for name, rgbd in [
    ["rgbd_regular", rgbd_regular],
    ["rgbd_random", rgbd_random],
    ["rgbd_cond", rgbd_cond],
    ["rgbd_gen_eval", rgbd_gen_eval],
]:
    rgbd.save_as(
        out_dir=os.path.join(data_dir, name),
        overwrite=True,
        mode="png",
        background_color=0,
        flag_save_space=True,
    )

In [ ]:
# save to a local dir to be tarred
tar_dir = os.path.join(out_root_dir, "tar_results")
tar_utils.save_one_sample_from_rendered_rgbds(
    data_dir=data_dir,
    out_dir=tar_dir,
    uid="bunny",
    new_uid="bunny_r0000",
    num_points=3_000_000,
    num_random_views=2,
    num_cond_views=2,
    num_gen_eval_views=2,
    keep_normal=False,
    rgbd_save_format="png",
    seed=0,
)

In [ ]:
# create tar
# you can save multiple samples into `tar_results`.
# depending on network speed, ideally a tar should be around 1-2 GB

tar_id = "example"  # uuid.uuid4()
tar_name = f"{tar_id}.tar"

# try to sort by file name (might not work on mac)
cmd = f"cd {tar_dir} && tar --sort=name -cf ../{tar_name} ."
print(cmd, flush=True)
return_code = os.system(cmd)

if return_code != 0:
    cmd = f"cd {tar_dir} && tar -cf ../{tar_name} ."
    print(cmd, flush=True)
    return_code = os.system(cmd)

if return_code == 0:
    print(f"Successfully saved {tar_name}")

In [ ]:
# (optionally) save points for tokenizer example
st_pcd = rgbd_regular.get_pcd(
    subsample=1,
    compute_ray_feature=True,
)
st_pcd = st_pcd.extract_valid_point_cloud(bidx=0)
point_xyz_w = st_pcd.xyz_w.squeeze(0)  # (n, 3xyz_w)  [-1, 1]
point_rgb = st_pcd.rgb.squeeze(0)  # (n, 3rgb) [0, 1]
point_view_dir = st_pcd.captured_view_direction_w.squeeze(0)  # (n, 3xyz_w) normalized, from pinhole to point

# slightly shrinks the points to avoid boundary issue for marching cube
point_xyz_w = point_xyz_w * 0.9

ridxs = np.random.permutation(len(point_xyz_w))[: 2**20]

pdict = dict(
    point_xyz_w=point_xyz_w[ridxs].cpu().numpy(),  # (n, 3xyz_w)  [-1, 1]
    point_rgb=(point_rgb[ridxs] * 255).cpu().numpy().astype(np.uint8),  # (n, 3rgb) [0, 1]
    point_view_dir=(((point_view_dir[ridxs] + 1) * 0.5) * 255).cpu().numpy().astype(np.uint8),  # (n, 3xyz_w)
)

filename = "assets/bunny.npz"
os.makedirs(os.path.dirname(filename), exist_ok=True)
np.savez_compressed(
    filename,
    **pdict,
)